# 1.1 Verosimilitud de un cliente

$$L(\lambda,p \mid x,t_x,T) = (1-p)^x \lambda^x e^{-\lambda T} + \delta_{x>0}\, p(1-p)^{x-1}\lambda^x e^{-\lambda t_x}$$

Hay dos historias que producen los mismos datos $(x, t_x, T)$:

**1. El cliente sigue vivo al final de la calibración (primer término).**
El factor $(1-p)^x$ es la probabilidad de que el cliente no haya muerto después de ninguna de sus $x$ compras repetidas. El factor $\lambda^x e^{-\lambda T}$ es la verosimilitud de Poisson de observar $x$ compras en $(0, T]$. Como $e^{-\lambda T} = e^{-\lambda t_x}\, e^{-\lambda (T - t_x)}$, incluye que el proceso corrió hasta $t_x$ y que no hubo compras entre $t_x$ y $T$.

**2. El cliente murió justo después de su última compra, en $t_x$ (segundo término).**
El cliente sobrevivió a las primeras $x-1$ compras repetidas, con probabilidad $(1-p)^{x-1}$, y murió justo después de la $x$-ésima, con probabilidad $p$. Como ya no compra después de morir, el proceso de Poisson solo cuenta hasta $t_x$, y por eso aparece $e^{-\lambda t_x}$ en lugar de $e^{-\lambda T}$.

**3. Por qué el segundo término no existe cuando $x = 0$.**
En el BG/NBD el cliente solo puede morir justo después de una compra **repetida**. Si $x = 0$, el cliente solo hizo su compra inicial y no hubo ningún momento en que pudiera morir, así que la única historia posible es que sigue vivo. Por eso $\delta_{x>0} = 0$ y solo queda el primer término. (Esta es la pieza que cambia el MBG/NBD en 4.6.)

# 1.2 La mezcla

Como $\lambda$ y $p$ son independientes (supuesto v), cada término se integra por separado: $\lambda$ con la densidad $\text{Gamma}(r,\alpha)$ y $p$ con la densidad $\text{Beta}(a,b)$.

**Primer término** (cliente vivo en $T$):
- En $\lambda$: $\lambda^{x}e^{-\lambda T}\cdot\frac{\alpha^r}{\Gamma(r)}\lambda^{r-1}e^{-\alpha\lambda} \Rightarrow \frac{\Gamma(r+x)\,\alpha^r}{\Gamma(r)\,(\alpha+T)^{r+x}}$
- En $p$: $(1-p)^x\cdot\frac{p^{a-1}(1-p)^{b-1}}{B(a,b)} \Rightarrow \frac{B(a,\,b+x)}{B(a,b)}$

**Segundo término** (murió en $t_x$, solo si $x>0$):
- En $\lambda$: igual, con $t_x$ en lugar de $T$: $\frac{\Gamma(r+x)\,\alpha^r}{\Gamma(r)\,(\alpha+t_x)^{r+x}}$
- En $p$: $p\,(1-p)^{x-1}\cdot p^{a-1}(1-p)^{b-1} = p^{a}(1-p)^{b+x-2} \Rightarrow \frac{B(a+1,\,b+x-1)}{B(a,b)}$

**Resultado:**
$$L(r,\alpha,a,b \mid x,t_x,T)=\frac{\Gamma(r+x)\,\alpha^r}{\Gamma(r)}\left[\frac{B(a,b+x)}{B(a,b)}\frac{1}{(\alpha+T)^{r+x}}+\delta_{x>0}\frac{B(a+1,b+x-1)}{B(a,b)}\frac{1}{(\alpha+t_x)^{r+x}}\right]$$

In [ ]:
import sys; sys.path.append("..")
from src.data_prep import cargar_transacciones, construir_resumen
from src.bgnbd import log_verosimilitud

df = cargar_transacciones("../data/raw/CDNOW_sample.txt")
res = construir_resumen(df)
x, t_x, T = res["x"].values, res["t_x"].values, res["T"].values

ll = log_verosimilitud([0.243, 4.414, 0.793, 2.426], x, t_x, T)
print(f"Log-verosimilitud: {ll:.2f}")

In [ ]:
import numpy as np
from scipy.optimize import minimize

def neg_ll(log_params):
    params = np.exp(log_params)                     # siempre positivos
    return -log_verosimilitud(params, x, t_x, T)    # minimizamos el negativo

inicios = [np.zeros(4), np.array([-1, 1, -0.5, 1]), np.ones(4)]

resultados = []
for inicio in inicios:
    res_opt = minimize(neg_ll, inicio, method="Nelder-Mead",
                       options={"maxiter": 5000, "xatol": 1e-8, "fatol": 1e-8})
    resultados.append(res_opt)
    print(np.exp(res_opt.x).round(4), -res_opt.fun, res_opt.success)

mejor = min(resultados, key=lambda o: o.fun)
r_hat, alpha_hat, a_hat, b_hat = np.exp(mejor.x)
ll_max = -mejor.fun
print(f"\nr={r_hat:.3f}  alpha={alpha_hat:.3f}  a={a_hat:.3f}  b={b_hat:.3f}  LL={ll_max:.1f}")

# 2.1 Ajuste a CDNOW

Se estimaron $(r,\alpha,a,b)$ maximizando la log-verosimilitud con los datos de calibración (39 semanas). Se optimizó sobre los logaritmos de los parámetros, con Nelder–Mead, desde tres puntos de partida distintos. Los tres convergieron al mismo resultado.

| Parámetro | Estimado (este trabajo) | Fader, Hardie y Lee (2005) |
|---|---|---|
| $r$ | 0.243 | 0.243 |
| $\alpha$ | 4.414 | 4.414 |
| $a$ | 0.793 | 0.793 |
| $b$ | 2.426 | 2.426 |
| Log-verosimilitud | −9582.4 | −9582.4 |

Los estimados coinciden con los del artículo, lo que valida la preparación de los datos $(x, t_x, T)$ y la implementación de la verosimilitud.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gamma, beta

print("Tasa de compra promedio (por semana):", round(r_hat / alpha_hat, 4))
print("Prob. media de morir tras una compra:", round(a_hat / (a_hat + b_hat), 4))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

lam = np.linspace(0.001, 0.5, 400)
ax[0].plot(lam, gamma.pdf(lam, a=r_hat, scale=1 / alpha_hat))   # scale = 1/tasa
ax[0].set_ylim(0, 15)
ax[0].set_title("Heterogeneidad en la tasa de compra λ")
ax[0].set_xlabel("λ (compras por semana)")

p = np.linspace(0.001, 0.999, 400)
ax[1].plot(p, beta.pdf(p, a_hat, b_hat))
ax[1].set_ylim(0, 8)
ax[1].set_title("Heterogeneidad en la prob. de morir p")
ax[1].set_xlabel("p")

plt.tight_layout()
plt.show()

# 2.2 Interpretación de los parámetros

- **Tasa de compra promedio de un cliente activo:** $r/\alpha = 0.243/4.414 \approx 0.055$ compras por semana, es decir, una compra cada ~18 semanas.
- **Probabilidad promedio de morir después de una compra:** $a/(a+b) = 0.793/3.219 \approx 0.246$ (24.6%).

**Qué dicen las densidades sobre qué tan parecidos son los clientes:**

Los clientes son muy distintos entre sí. Como $r<1$, la densidad de $\lambda$ es decreciente y se dispara cerca de 0: aproximadamente la mitad de los clientes tiene $\lambda<0.01$ (menos de una compra cada 100 semanas) y cerca del 73% tiene $\lambda<0.05$. En el otro extremo, solo ~6% compra más de una vez cada 4 semanas ($\lambda>0.25$). Por eso la media (0.055) no representa al cliente típico: la mayoría está muy por debajo y una minoría muy activa la sube.

Con la probabilidad de morir pasa algo parecido. Como $a<1$, la densidad de $p$ también se concentra cerca de 0: ~32% de los clientes tiene $p<0.1$, y solo ~14% tiene $p>0.5$. Hay muchos clientes "leales" que casi no abandonan y una minoría con alta probabilidad de irse tras cada compra.

En resumen, no existe un "cliente promedio": la base mezcla una mayoría de compradores muy esporádicos con una minoría activa, y eso es lo que el modelo captura con las distribuciones gamma y beta.